# 日本語の述語項構造解析をする (KWJA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/16_semantics/pas_kwja.ipynb)

NLP クックブック 16-01 ／ 意味解析

各述語について、ガ格・ヲ格・ニ格に当たる語を求めたい。省略された主語や目的語（ゼロ代名詞）も補いたい。

In [1]:
!pip install -q kwja

In [2]:
import subprocess
from rhoknp import Document

def kwja(text, model_size="base"):
    """KWJA を実行して、解析結果を rhoknp の Document として返す (初回はモデルのダウンロードに数分かかる)"""
    result = subprocess.run(["kwja", "--text", text, "--tasks", "char,word", "--model-size", model_size],
                            capture_output=True, text=True, check=True)
    return Document.from_knp(result.stdout)

doc = kwja("太郎は書店で新しい辞書を買った。家に帰ってから、花子に見せた。")
for sent in doc.sentences:
    print(sent.text)
    for bp in sent.base_phrases:
        args = bp.pas.get_all_arguments(relax=False) if bp.pas else {}
        if args:
            print("  述語:", bp.text, {case: [str(a) for a in arguments] for case, arguments in args.items()})

太郎は書店で新しい辞書を買った。
  述語: 新しい {'ガ': ['辞書を']}
  述語: 買った。 {'ガ': ['太郎は'], 'ヲ': ['辞書を'], 'デ': ['書店で']}
家に帰ってから、花子に見せた。
  述語: 家に {'ノ': ['太郎は']}
  述語: 帰ってから、 {'ガ': ['太郎は'], 'ニ': ['家に']}
  述語: 見せた。 {'ガ': ['太郎は'], 'ヲ': ['辞書を'], 'ニ': ['花子に']}


In [3]:
# 項の種類を見る: 係り受けがある項 / 文内・文間のゼロ照応 / 文章の外を指す外界照応 (著者・読者・不特定の人)
from rhoknp.cohesion import EndophoraArgument, ExophoraArgument

for bp in doc.base_phrases:
    if not bp.pas:
        continue
    for case, arguments in bp.pas.get_all_arguments(relax=False).items():
        for a in arguments:
            if isinstance(a, EndophoraArgument):
                where = "同じ文" if a.base_phrase.sentence.sid == bp.sentence.sid else "前の文"
                print(f"{bp.text} の{case}格 = {a.base_phrase.text} ({a.type.value}, {where})")
            elif isinstance(a, ExophoraArgument):
                print(f"{bp.text} の{case}格 = {a.exophora_referent.text} (外界照応)")

新しい のガ格 = 辞書を (N, 同じ文)
買った。 のガ格 = 太郎は (N, 同じ文)
買った。 のヲ格 = 辞書を (C, 同じ文)
買った。 のデ格 = 書店で (C, 同じ文)
家に のノ格 = 太郎は (O, 前の文)
帰ってから、 のガ格 = 太郎は (O, 前の文)
帰ってから、 のニ格 = 家に (C, 同じ文)
見せた。 のガ格 = 太郎は (O, 前の文)
見せた。 のヲ格 = 辞書を (O, 前の文)
見せた。 のニ格 = 花子に (C, 同じ文)


- 2 文目の「帰って」「見せた」には主語が書かれていないが、前の文の「太郎」がガ格として補われる。これがゼロ照応解析。
- KWJA は京都大学で開発された統合的な日本語解析器（BERT 系のエンコーダを使う）。形態素解析から、固有表現、係り受け、述語項構造、共参照、談話関係までを一度に行う。
- `--model-size` は tiny / base / large。base 以上は CPU だと 1 文に数秒かかる。大量に処理するなら GPU を使う。
- インストール時に Transformers のバージョンが入れ替わることがある。import でエラーになったらランタイムを再起動する。